In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from rich.table import Table

from src.database_helpers import DatabaseHelper

In [ ]:
dpath_data = Path("../data")
dpath_schema = dpath_data / "schema"
dpath_processed = dpath_data / "processed"
dpath_raw = dpath_data / "raw"
subsets = [
    "all",
    "psychoactive",
    "healthy",
    "hypertension",
]

fpath_Xy = dpath_processed / "Xy.pkl"
fpath_udis = dpath_raw / "UDIs.csv"
fpath_demographic = dpath_processed / "demographic.csv"
fpath_demographic_instance0 = (
    dpath_processed / "demographics_table" / "demographic-instance0.csv"
)
fpaths_samples = {
    subset: dpath_processed
    / f"bootstrap_samples-{subset}-50_20257_30steps_2000times.pkl"
    for subset in subsets
}

col_sex = "31-0.0"
col_age = "21003-2.0"
col_ethnicity = "21000-0.0"
col_weight = "21002-2.0"
col_handedness = "1707-0.0"

cols_to_keep = [
    col_sex,
    col_age,
    col_ethnicity,
    col_weight,
    col_handedness,
]

ETHNICITY_THOUSANDS_MAPPING = {
    1: "White",
    2: "Mixed",
    3: "Asian or Asian British",
    4: "Black or Black British",
    5: "Chinese",
    6: "Other",
}

col_subset = "subset"

In [ ]:
with fpath_Xy.open("rb") as file_data:
    Xy = pickle.load(file_data)

samples_by_subset = {
    subset: pickle.load(fpath_samples.open("rb"))
    for subset, fpath_samples in fpaths_samples.items()
}

In [ ]:
participants_by_subset = {subset: set() for subset in subsets}
for subset, samples in samples_by_subset.items():
    for sample in samples.i_samples_val_all:
        participants_by_subset[subset].update(sample)
    print(f"{subset}: {len(participants_by_subset[subset])}")

In [ ]:
# get overlaps
for subset1 in subsets:
    for subset2 in subsets:
        if subset1 == subset2:
            break
        if subset1 == "all" or subset2 == "all":  # full overlaps expected for these
            continue
        print(
            f"{subset1} vs {subset2}: {len(participants_by_subset[subset1].intersection(participants_by_subset[subset2]))}"
        )

In [ ]:
db_helper = DatabaseHelper(dpath_schema=dpath_schema, fpath_udis=fpath_udis)


def process_ethnicity(value):
    for thousands, ethnicity in ETHNICITY_THOUSANDS_MAPPING.items():
        if value == thousands or value // 1000 == thousands:
            return ethnicity
    return np.nan


df_demographic = pd.concat(
    [
        pd.read_csv(fpath_demographic, index_col=0),
        pd.read_csv(fpath_demographic_instance0, index_col=0),
    ],
    axis="columns",
)

df_demographic = df_demographic[cols_to_keep]
# df_demographic.columns = db_helper.udis_to_text(df_demographic.columns)

# for col in df_demographic.columns:
#     print(col)

# ethnicity
df_demographic[col_ethnicity] = df_demographic[col_ethnicity].apply(process_ethnicity)

df_demographic

In [ ]:
# for col in df_demographic:
#     print(col)
#     print(df_demographic[col].value_counts())
#     print()

In [ ]:
dfs_for_table = {
    subset: df_demographic.iloc[list(participants_by_subset[subset])]
    for subset in subsets
}

In [ ]:
from typing import Iterable

TAB_SPACES = "    "


def cohort_summary(dfs_by_subset, title):
    def gen_row(D, *, agg, col, f=".1f", sep=" ± "):
        if not isinstance(agg, str) and isinstance(agg, Iterable):
            return [sep.join([f"{d.loc[a][col]:{f}}" for a in agg]) for d in D]
        else:
            return [f"{d.loc[agg][col]:{f}}" for d in D]

    def ratio(series):
        count = sum(series)
        return f"{count:.0f} ({count / len(series) * 100:.1f}%)"

    dfs_described = [df.describe() for df in dfs_by_subset.values()]

    table = Table(title=title, show_footer=True)

    table.add_column("Subject groups", footer="Values expressed as mean ± SD.")

    for subset in subsets:
        table.add_column(subset.capitalize())

    table.add_row("N", *gen_row(dfs_described, agg="count", col=col_age, f=".0f"))

    table.add_row(
        "Age (years)", *gen_row(dfs_described, agg=["mean", "std"], col=col_age)
    )
    # table.add_row(
    #     "Age range (years)", *gen_row(D, agg=["min", "max"], col=col_age, sep=" - ")
    # )

    table.add_row(
        "Weight (kg)", *gen_row(dfs_described, agg=["mean", "std"], col=col_weight)
    )

    table.add_row("Sex")
    table.add_row(
        f"{TAB_SPACES}Male", *[ratio(df[col_sex] == 1) for df in dfs_by_subset.values()]
    )
    table.add_row(
        f"{TAB_SPACES}Female",
        *[ratio(df[col_sex] == 0) for df in dfs_by_subset.values()],
    )

    table.add_row("Ethnicity")
    for ethnicity in ETHNICITY_THOUSANDS_MAPPING.values():
        table.add_row(
            f"{TAB_SPACES}{ethnicity}",
            *[ratio(df[col_ethnicity] == ethnicity) for df in dfs_by_subset.values()],
        )

    table.add_row("Handedness")
    table.add_row(
        f"{TAB_SPACES}Right",
        *[ratio(df[col_handedness] == 1) for df in dfs_by_subset.values()],
    )
    table.add_row(
        f"{TAB_SPACES}Left",
        *[ratio(df[col_handedness] == 2) for df in dfs_by_subset.values()],
    )
    table.add_row(
        f"{TAB_SPACES}Both",
        *[ratio(df[col_handedness] == 3) for df in dfs_by_subset.values()],
    )

    return table


cohort_summary(dfs_for_table, title="Demographic information by cohort")

In [ ]:
df_demographic[col_weight].value_counts(dropna=False)  # .sort_index()